# Proyecto Neobank


**1. Análisis de Datos**


In [ ]:
#Importación de  librerías para manipulación y análisis de datos
import pandas as pd
import numpy as np
from datetime import datetime

#Importación de librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [ ]:
#Users

# from google.cloud import bigquery

# client = bigquery.Client()
# query = """
#     SELECT * FROM `numeric-advice-452700-j9.neo_bank_.users`
# """
# users = client.query(query).to_dataframe()

# users

users=pd.read_csv('../data/tablas-actualizadas/users.csv')
users.info()

In [ ]:
users.head()

In [ ]:
users

In [ ]:
users.isnull().sum()

In [ ]:
#Devices
# query = """
#     SELECT * FROM `numeric-advice-452700-j9.neo_bank_.devices`
# """
# devices = client.query(query).to_dataframe()

devices=pd.read_csv('../data/tablas-actualizadas/devices.csv')
devices.info()

In [ ]:
devices.head()

In [ ]:
devices.isnull().sum()

In [ ]:
#Notifications
# query = """
#     SELECT * FROM `numeric-advice-452700-j9.neo_bank_.notifications`
# """
# notifications = client.query(query).to_dataframe()

notifications=pd.read_csv('../data/tablas-actualizadas/notifications.csv')
notifications.info()

In [ ]:
notifications.head()

In [ ]:
notifications.isnull().sum()

In [ ]:
# #Transactions
# query = """
#     SELECT * FROM `numeric-advice-452700-j9.neo_bank_.transactions`
# """
# transactions = client.query(query).to_dataframe()

transactions=pd.read_csv('../data/tablas-actualizadas/transactions.csv')
transactions.info()

In [ ]:
transactions.head()

In [ ]:
transactions.isnull().sum()

**1.1 Transformado variables**


In [ ]:
# Transformando Variables fecha

users['created_date']=pd.to_datetime(users['created_date'], errors='coerce')
transactions['created_date']=pd.to_datetime(transactions['created_date'], errors='coerce')
notifications['created_date']=pd.to_datetime(notifications['created_date'], errors='coerce')

users['created_date'], transactions['created_date'], notifications['created_date']

In [ ]:
#Última actividad

ultima_actividad=transactions['created_date'].max()
ultima_actividad.date()


In [ ]:
f'La última fecha de actividad registrada es el {ultima_actividad.date()}'

In [ ]:
#Ventanas de inactividad
sesenta_dias=pd.to_timedelta(60,unit='d')
noventa_dias=pd.to_timedelta(90,unit='d')
cientoveinte_dias=pd.to_timedelta(120,unit='d')

inactividad_60=ultima_actividad-sesenta_dias
inactividad_90=ultima_actividad-noventa_dias
inactividad_120=ultima_actividad-cientoveinte_dias

inactividad_60.date(), inactividad_90.date(), inactividad_120.date()

**1.2 Calculo del churn**

In [ ]:
#Casos a analizar

#Caso A

f' Cualquier usuario con una trasacción anterior a {inactividad_60.date()} será considerado fugado'

In [ ]:
transactions.groupby('user_id')['created_date'].max()

In [ ]:
ultima_transaccion_por_usuario=transactions.groupby('user_id')['created_date'].max().reset_index()
ultima_transaccion_por_usuario

In [ ]:
ultima_transaccion_por_usuario.rename(columns={'created_date':'ultima_transaccion'},inplace=True)
ultima_transaccion_por_usuario

In [ ]:
#Confección de DataFrame con información

df_users=pd.merge(users,ultima_transaccion_por_usuario,how='left',on='user_id')
df_users.info()

In [ ]:
df_users.isnull().sum()

In [ ]:
# Creando la variable "Churn"

# CASO A

df_users_churn_a = df_users


In [ ]:
df_users_churn_a['churn']=((df_users['ultima_transaccion']<inactividad_60)|(df_users['ultima_transaccion'].isnull())).astype(int)
df_users_churn_a

In [ ]:
df_users_churn_a.isnull().sum()

In [ ]:
df_users_churn_a['churn'].value_counts(normalize=True), df_users_churn_a['churn'].value_counts()

In [ ]:
#CASOS B & C

df_users_churn_b=df_users
df_users_churn_c=df_users

df_users_churn_b['churn']=((df_users['ultima_transaccion']<inactividad_90)|(df_users['ultima_transaccion'].isnull())).astype(int)
df_users_churn_c['churn']=((df_users['ultima_transaccion']<inactividad_120)|(df_users['ultima_transaccion'].isnull())).astype(int)


In [ ]:
df_users_churn_b['churn'].value_counts(normalize=True), df_users_churn_b['churn'].value_counts()

In [ ]:
df_users_churn_b.isnull().sum()

In [ ]:
df_users_churn_c['churn'].value_counts(normalize=True), df_users_churn_c['churn'].value_counts()

In [ ]:
df_users_churn_c.isnull().sum()

**1.3 Modificando la tabla usuarios**

In [ ]:
df_users_churn_a['age']= ultima_actividad.year - df_users_churn_a['birth_year']
df_users_churn_a['age']

In [ ]:
df_users_churn_b['age']= ultima_actividad.year - df_users_churn_b['birth_year']
df_users_churn_c['age']= ultima_actividad.year - df_users_churn_c['birth_year']

df_users_churn_b['age'], df_users_churn_c['age']

In [ ]:
df_users_churn_a['antiguedad']= (ultima_actividad - df_users_churn_a['created_date']).dt.days
df_users_churn_b['antiguedad']= (ultima_actividad - df_users_churn_b['created_date']).dt.days
df_users_churn_c['antiguedad']= (ultima_actividad - df_users_churn_c['created_date']).dt.days

# Dias desde la última transacción

df_users_churn_a['dias_ultima_transaccion']= (ultima_actividad - df_users_churn_a['ultima_transaccion']).dt.days
df_users_churn_b['dias_ultima_transaccion']= (ultima_actividad - df_users_churn_b['ultima_transaccion']).dt.days
df_users_churn_c['dias_ultima_transaccion']= (ultima_actividad - df_users_churn_c['ultima_transaccion']).dt.days

#Rellenado valores sin registro de ultima trasacción (999)

df_users_churn_a['dias_ultima_transaccion'].fillna(999,inplace=True)
df_users_churn_b['dias_ultima_transaccion'].fillna(999,inplace=True)
df_users_churn_c['dias_ultima_transaccion'].fillna(999,inplace=True)


In [ ]:
df_users_churn_a.info()

In [ ]:
df_users_churn_b.info()

In [ ]:
df_users_churn_c.info()

**1.4 Modificación General de Tablas**

In [ ]:
#Modificando variables de Tabla Transacciones

estadisticas_transacciones=transactions.groupby('user_id').agg(
    total_transacciones=('transaction_id','count'),total_monto=('amount_usd','sum'),promedio_monto=('amount_usd','mean'),nb_transacciones_declinadas=('transactions_state',lambda x: (x=='DECLINED').sum()),nb_transacciones_reversadas=('transactions_state',lambda x: (x=='REVERSED').sum()),nb_transacciones_completadas=('transactions_state',lambda x: (x=='COMPLETED').sum())).reset_index()


estadisticas_transacciones

In [ ]:
#Cálculo de ratio de transacciones

estadisticas_transacciones['ratio_transacciones_reversadas']=estadisticas_transacciones['nb_transacciones_reversadas']/estadisticas_transacciones['total_transacciones']
estadisticas_transacciones['ratio_transacciones_completadas']=estadisticas_transacciones['nb_transacciones_completadas']/estadisticas_transacciones['total_transacciones']
estadisticas_transacciones['ratio_transacciones_declinadas']=estadisticas_transacciones['nb_transacciones_declinadas']/estadisticas_transacciones['total_transacciones']

estadisticas_transacciones

In [ ]:
#Union de nuevas variables para 'Transacciones' con DataFrame de información

df_users_churn_a=pd.merge(df_users_churn_a,estadisticas_transacciones,how='left',on='user_id')
df_users_churn_b=pd.merge(df_users_churn_b,estadisticas_transacciones,how='left',on='user_id')
df_users_churn_c=pd.merge(df_users_churn_c,estadisticas_transacciones,how='left',on='user_id')

In [ ]:
#Modificando variables de Tabla Notificaciones

estadisticas_notificaciones=notifications.groupby('user_id').agg(total_notificaciones=('user_id','count')).reset_index()
estadisticas_notificaciones

In [ ]:
#Union de nuevas variables para 'Notificaciones' con DataFrame de información

df_users_churn_a=pd.merge(df_users_churn_a,estadisticas_notificaciones,how='left',on='user_id')
df_users_churn_b=pd.merge(df_users_churn_b,estadisticas_notificaciones,how='left',on='user_id')
df_users_churn_c=pd.merge(df_users_churn_c,estadisticas_notificaciones,how='left',on='user_id')

In [ ]:
df_users_churn_a.info()

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
#Modificando variables de Tabla Dispositivos

devices.rename(columns={'string_field_0':'marca','string_field_1':'user_id'},inplace=True)
devices


In [ ]:
estadisticas_devices=devices.groupby('user_id').agg(nb_devices=('marca','nunique')).reset_index()
estadisticas_devices

In [ ]:
#Union de nuevas variables para 'Dispositivos' con DataFrame de información

df_users_churn_a=pd.merge(df_users_churn_a,estadisticas_devices,how='left',on='user_id')
df_users_churn_b=pd.merge(df_users_churn_b,estadisticas_devices,how='left',on='user_id')
df_users_churn_c=pd.merge(df_users_churn_c,estadisticas_devices,how='left',on='user_id')

In [ ]:
df_users_churn_a.isnull().sum()

In [ ]:
#Nulos Transacciones
porcentaje_nulos_transacciones=(df_users_churn_a['total_transacciones'].isnull().sum()/len(df_users_churn_a))*100
print(f"El porcentaje de nulos en '{'total_transacciones'}' es: {porcentaje_nulos_transacciones:.2f}%")

In [ ]:
#Nulos notificaciones Push / email en tabla de usuarios

porcentaje_nulos_notificaciones=(df_users_churn_a['attributes_notifications_marketing_push'].isnull().sum()/len(df_users_churn_a))*100
print(f"El porcentaje de nulos en '{'attributes_notifications_marketing_push'}' es: {porcentaje_nulos_notificaciones:.2f}%")

In [ ]:
attributes_notifications_marketing_push_nulos=df_users_churn_a['attributes_notifications_marketing_push'].isnull()
attributes_notifications_marketing_push_nulos

In [ ]:
usuarios_marketing_push_nulos=df_users_churn_a.loc[attributes_notifications_marketing_push_nulos, 'user_id']
usuarios_marketing_push_nulos

In [ ]:
attributes_notifications_marketing_email_nulos=df_users_churn_a['attributes_notifications_marketing_email'].isnull()
usuarios_marketing_email_nulos=df_users_churn_a.loc[attributes_notifications_marketing_email_nulos, 'user_id']
usuarios_marketing_email_nulos

In [ ]:
validación_usuarios_nulos_campana=usuarios_marketing_push_nulos.isin(usuarios_marketing_email_nulos)
validación_usuarios_nulos_campana

In [ ]:
#Asignar valor a usuarios sin campanas de marketing o email =0

df_users_churn_a['attributes_notifications_marketing_push'].fillna(0,inplace=True)
df_users_churn_a['attributes_notifications_marketing_email'].fillna(0,inplace=True)
df_users_churn_b['attributes_notifications_marketing_push'].fillna(0,inplace=True)
df_users_churn_b['attributes_notifications_marketing_email'].fillna(0,inplace=True)
df_users_churn_c['attributes_notifications_marketing_push'].fillna(0,inplace=True)
df_users_churn_c['attributes_notifications_marketing_email'].fillna(0,inplace=True)

In [ ]:
df_users_churn_a.isnull().sum()

In [ ]:
#Eliminación de usuarios que nunca realizaron una transacción

df_users_churn_a.dropna(subset=['ultima_transaccion'],inplace=True)
df_users_churn_a.isnull().sum()

In [ ]:
df_users_churn_b.dropna(subset=['ultima_transaccion'],inplace=True)
df_users_churn_c.dropna(subset=['ultima_transaccion'],inplace=True)

In [ ]:
#Asignar valor a usuarios que nunca fueron notificados y si realizaron transacciones =0

df_users_churn_a['total_notificaciones'].fillna(0,inplace=True)
df_users_churn_b['total_notificaciones'].fillna(0,inplace=True)
df_users_churn_c['total_notificaciones'].fillna(0,inplace=True)

In [ ]:
df_users_churn_a.info()

In [ ]:
#Porcantaje de la muestra Original
porcentaje_muestra_original=(len(df_users_churn_a)/len(users))*100
print(f"El porcentaje de la muestra original es: {porcentaje_muestra_original:.2f}%")

In [ ]:
#Balance del Churn

df_users_churn_a['churn'].value_counts(normalize=True), df_users_churn_a['churn'].value_counts()

In [ ]:
df_users_churn_b['churn'].value_counts(normalize=True), df_users_churn_b['churn'].value_counts()

In [ ]:
df_users_churn_c['churn'].value_counts(normalize=True), df_users_churn_c['churn'].value_counts()

Nota: Tras la limpieza del DataFrame ya no hay diferencias entre diversas ventanas de inactividad transaccional, por lo tanto se avanzará solo con **df_users_churn_a**

In [ ]:
#Eliminacion del id de usuario

df_users_churn_a.drop(columns='user_id',inplace=True)

In [ ]:
#Eliminación de created_date y ultima trasanccion

df_users_churn_a.drop(columns=['created_date','ultima_transaccion'],inplace=True)

### **1.5 Entrenamiento del Modelo**

In [ ]:
## Importación de herramientas de Scikit-Learn para modelamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
#Instalación de Pycaret
# !pip install pycaret

In [ ]:
# !pip freeze | grep pycaret

In [ ]:
variables_categóricas=df_users_churn_a.select_dtypes(include=['object']).columns.tolist()
print(f"Variables categóricas: {variables_categóricas}")

In [ ]:
#Número de registros distintos para variables categóricas

df_users_churn_a[variables_categóricas].nunique()

In [ ]:
#Grafico de Distribución por ciudad

import plotly.express as px
fig_1= px.histogram(df_users_churn_a, x='city')
fig_1.show()

In [ ]:
df_users_churn_a['city'].value_counts()

In [ ]:
#Grafico de Distribución por país.
fig_2= px.histogram(df_users_churn_a, x='country')
fig_2.show()

In [ ]:
df_users_churn_a['country'].value_counts()

In [ ]:
#Eliminación de variable ciudad y pais

df_users_churn_a.drop(columns=['city','country', 'Unnamed: 0'],inplace=True)
#Eliminando variables : día ultima transacción, ratio_transacciones_reversadas, ratio_transacciones_completadas,ratio_transacciones_declinadas.

df_users_churn_a.drop(columns=['dias_ultima_transaccion','ratio_transacciones_reversadas','ratio_transacciones_completadas','ratio_transacciones_declinadas'],inplace=True)



In [ ]:
variables_categóricas_2=df_users_churn_a.select_dtypes(include=['object']).columns.tolist()
print(f"Variables categóricas: {variables_categóricas_2}")

In [ ]:
df_users_churn_a.info()

In [ ]:
X = df_users_churn_a.drop(columns=["churn"])
y = df_users_churn_a["churn"]

display(X, y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

# Separar columnas
categorical_cols = ["plan"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

In [ ]:
# Transformadores
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
numeric_transformer = StandardScaler()

In [ ]:
# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

In [ ]:
# Pipeline
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])


In [ ]:
pipeline.fit(X_train, y_train)


In [ ]:
score = pipeline.score(X_test, y_test)
print(f"Accuracy en set de prueba: {score:.4f}")


In [ ]:
from sklearn.metrics import classification_report

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
import joblib
# joblib.dump(pipeline, "best_model.pkl")
joblib.dump(pipeline, '../models/best_model2.pkl')
